## Read the artist_collab table

In [1]:
import pandas as pd
import networkx as nx
from node2vec import Node2Vec


### Ran the following query in PSQL

In [2]:
q = """
COPY (
    SELECT
        artist_id AS src,
        neighbor_artist_id AS dst,
        COUNT(*) AS weight
    FROM artist_collab
    GROUP BY artist_id, neighbor_artist_id
) TO '/tmp/artist_edges.csv' CSV HEADER;
"""

In [ ]:
artist_collab_n2v = pd.read_csv("/tmp/artist_edges.csv")

In [7]:
import pandas as pd
import numpy as np
from pecanpy import pecanpy as p2v
from sklearn.decomposition import PCA
import plotly.express as px

# ============================================================
# INPUT: artist_collab_n2v
# Required columns: src, dst, weight
# ============================================================

# -------------------------
# 1. Deduplicate edges
# -------------------------
edges_df = (
    artist_collab_n2v
    .dropna(subset=["src", "dst"])
    .astype({"src": str, "dst": str})
    .groupby(["src", "dst"], as_index=False)
    .agg({"weight": "sum"})
)

# -------------------------
# 2. CREATE ARTIST ↔ INDEX MAPPING (CRITICAL)
# -------------------------
artists = pd.Index(
    pd.unique(pd.concat([edges_df["src"], edges_df["dst"]]))
)

artist_to_idx = {a: i for i, a in enumerate(artists)}
idx_to_artist = artists.to_numpy()

# Persist mapping
np.save("/tmp/idx_to_artist.npy", idx_to_artist)

print(f"Mapped {len(idx_to_artist)} artists")

# -------------------------
# 3. Rewrite edge list with integer indices
# -------------------------
edges_df["src_i"] = edges_df["src"].map(artist_to_idx)
edges_df["dst_i"] = edges_df["dst"].map(artist_to_idx)

edge_path = "/tmp/artist_edges.edgelist"
edges_df[["src_i", "dst_i", "weight"]].to_csv(
    edge_path,
    sep=" ",
    header=False,
    index=False
)

print(f"Wrote {len(edges_df)} edges to {edge_path}")

# -------------------------
# 4. Load graph into PecanPy
# -------------------------
g = p2v.SparseOTF(
    p=1.0,
    q=1.0,
    workers=2,
    verbose=True
)

g.read_edg(
    edge_path,
    weighted=True,
    directed=False,
    delimiter=" "
)

print("Graph loaded")
print("Nodes:", g.num_nodes)
print("Edges:", g.num_edges)

assert g.num_nodes == len(idx_to_artist)

# -------------------------
# 5. Train Node2Vec (embeddings returned here)
# -------------------------
out = g.embed(
    dim=32,
    num_walks=5,
    walk_length=10,
    window_size=5,
    epochs=1
)

print("Embedding matrix shape:", out.shape)

# -------------------------
# 6. Assign embeddings back to artist IDs
# -------------------------
df_emb = pd.DataFrame(
    out,
    columns=[f"emb_{i}" for i in range(out.shape[1])]
)
df_emb["artist_id"] = idx_to_artist

print("Final embedding table:", df_emb.shape)

# -------------------------
# 7. PCA + interactive visualization (sanity check)
# -------------------------
N_SAMPLE = min(20000, len(df_emb))
df_sample = df_emb.sample(n=N_SAMPLE, random_state=42)

X = df_sample.filter(like="emb_").values
X_2d = PCA(n_components=2, random_state=42).fit_transform(X)

df_sample["pc1"] = X_2d[:, 0]
df_sample["pc2"] = X_2d[:, 1]

fig = px.scatter(
    df_sample,
    x="pc1",
    y="pc2",
    hover_data=["artist_id"],
    title="Artist Collaboration Embeddings (Node2Vec / PecanPy)",
    opacity=0.6
)

fig.update_traces(marker=dict(size=4))
fig.update_layout(width=900, height=900)
fig.show()

# -------------------------
# 8. Save embeddings
# -------------------------
df_emb.to_csv("/tmp/artist_embeddings.csv", index=False)
print("Saved /tmp/artist_embeddings.csv")


Mapped 582964 artists
Wrote 3199860 edges to /tmp/artist_edges.edgelist
Graph loaded
Nodes: 582964
Edges: 3199860


  0%|          | 0/2914820 [00:00<?, ?it/s]

Embedding matrix shape: (582964, 32)
Final embedding table: (582964, 33)


Saved /tmp/artist_embeddings.csv


In [ ]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

# -----------------------------------
# DB connection (from env)
# -----------------------------------
DB_URL = os.getenv("PG_DSN")
if not DB_URL:
    raise RuntimeError("PG_DSN environment variable is not set")

# -----------------------------------
# Load embeddings
# -----------------------------------
df_emb = pd.read_csv("/tmp/artist_embeddings.csv")

# Ensure correct column order
emb_cols = [c for c in df_emb.columns if c.startswith("emb_")]
cols = ["artist_id"] + emb_cols
df_emb = df_emb[cols]

records = df_emb.values.tolist()

# -----------------------------------
# Build SQL
# -----------------------------------
insert_cols = ",".join(cols)
update_clause = ",".join([f"{c}=EXCLUDED.{c}" for c in emb_cols])

sql = f"""
INSERT INTO artist_embeddings_n2v ({insert_cols})
VALUES %s
ON CONFLICT (artist_id)
DO UPDATE SET
{update_clause}
"""

# -----------------------------------
# Execute
# -----------------------------------
with psycopg2.connect(DB_URL) as conn:
    with conn.cursor() as cur:
        execute_values(
            cur,
            sql,
            records,
            page_size=1000
        )

print(f"Inserted/updated {len(records)} artist embeddings")


Inserted/updated 582964 artist embeddings


In [ ]:
emb_df = pd.read_csv("/tmp/artist_embeddings.csv")
edge_path = "/tmp/artist_edges.edgelist"
edges_df = pd.read_csv(
    edge_path,
    sep=" ",
    header=None
)

edges_df.columns = ["src_i", "dst_i", "weight"]


In [10]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import umap
import plotly.graph_objects as go

# -----------------------------
# TUNABLE SAFETY PARAMETERS
# -----------------------------
MAX_NODES = 1500          # <- main stability knob
MAX_EDGES_PER_NODE = 3    # <- keeps structure without hairball
K_CLUSTERS = 10

# -----------------------------
# 1. EMBEDDINGS → DATAFRAME
# -----------------------------
emb_df = emb_df
# pd.DataFrame.from_dict(
#     embeddings_dict,
#     orient="index"
# )
emb_df.index.name = "node_id"
emb_df.reset_index(inplace=True)

# -----------------------------
# 2. SAMPLE NODES
# -----------------------------
if len(emb_df) > MAX_NODES:
    emb_df = emb_df.sample(MAX_NODES, random_state=42)

node_set = set(emb_df["node_id"])

# -----------------------------
# 3. FILTER EDGES (TOP-K PER NODE)
# -----------------------------
edges_filt = edges_df[
    edges_df["src"].isin(node_set) &
    edges_df["dst"].isin(node_set)
].copy()

# keep strongest edges per src if weight exists
if "weight" in edges_filt.columns:
    edges_filt = (
        edges_filt
        .sort_values("weight", ascending=False)
        .groupby("src")
        .head(MAX_EDGES_PER_NODE)
    )
else:
    edges_filt = edges_filt.groupby("src").head(MAX_EDGES_PER_NODE)

# -----------------------------
# 4. SCALE + PCA + UMAP
# -----------------------------
X = emb_df.drop(columns=["node_id"]).values
X_scaled = StandardScaler().fit_transform(X)

X_pca = PCA(n_components=20, random_state=42).fit_transform(X_scaled)

X_umap = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=42
).fit_transform(X_pca)

emb_df["x"] = X_umap[:, 0]
emb_df["y"] = X_umap[:, 1]

# -----------------------------
# 5. CLUSTERING
# -----------------------------
emb_df["cluster"] = KMeans(
    n_clusters=K_CLUSTERS,
    random_state=42
).fit_predict(X_pca)

# -----------------------------
# 6. EDGE COORDS (PLOTLY SAFE)
# -----------------------------
pos = emb_df.set_index("node_id")[["x", "y"]]

edge_x, edge_y = [], []
for _, r in edges_filt.iterrows():
    if r["src"] in pos.index and r["dst"] in pos.index:
        edge_x += [pos.loc[r["src"], "x"], pos.loc[r["dst"], "x"], None]
        edge_y += [pos.loc[r["src"], "y"], pos.loc[r["dst"], "y"], None]

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode="lines",
    line=dict(width=0.4, color="rgba(0,0,0,0.15)"),
    hoverinfo="none"
)

# -----------------------------
# 7. NODE TRACE
# -----------------------------
node_trace = go.Scatter(
    x=emb_df["x"],
    y=emb_df["y"],
    mode="markers",
    hovertext=emb_df["node_id"],
    hoverinfo="text",
    marker=dict(
        size=7,
        color=emb_df["cluster"],
        colorscale="Turbo",
        showscale=True,
        colorbar=dict(title="Cluster")
    )
)

# -----------------------------
# 8. PLOT
# -----------------------------
fig = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        title="Sampled PecanPy Graph (Interactive)",
        hovermode="closest",
        showlegend=False,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        margin=dict(l=20, r=20, t=40, b=20)
    )
)

fig.show()


/home/jonnym/.local/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [22]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import umap

# --------------------------------------------------
# SAFETY / SCALE CONTROLS
# --------------------------------------------------
MAX_EDGES = 3000        # primary knob (edges, not nodes)
MAX_EDGES_PER_NODE = 3
K_CLUSTERS = 10
RANDOM_STATE = 42

# --------------------------------------------------
# 1. LOAD DATA
# --------------------------------------------------
emb_df = pd.read_csv("/tmp/artist_embeddings.csv")

# enforce column names
edges_df = edges_df[["src_i", "dst_i", "weight"]]

# --------------------------------------------------
# 2. SAMPLE *KNOWN* EDGES
# --------------------------------------------------
edges_df = edges_df.sort_values("weight", ascending=False)

if len(edges_df) > MAX_EDGES:
    edges_df = edges_df.sample(MAX_EDGES, random_state=RANDOM_STATE)

# keep top-k edges per source to avoid hairball
edges_df = (
    edges_df
    .groupby("src_i", group_keys=False)
    .head(MAX_EDGES_PER_NODE)
)

# --------------------------------------------------
# 3. DERIVE NODE SET FROM EDGES
# --------------------------------------------------
node_ids = pd.unique(
    edges_df[["src_i", "dst_i"]].values.ravel()
)

# subset embeddings to only used nodes
emb_sub = emb_df.iloc[node_ids].copy()
emb_sub["node_id"] = node_ids

# --------------------------------------------------
# 4. EMBEDDINGS → PCA → UMAP
# --------------------------------------------------
X = emb_sub.drop(columns=["node_id"]).values

X_scaled = StandardScaler().fit_transform(X)

X_pca = PCA(
    n_components=min(20, X.shape[1]),
    random_state=RANDOM_STATE
).fit_transform(X_scaled)

X_umap = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=RANDOM_STATE
).fit_transform(X_pca)

emb_sub["x"] = X_umap[:, 0]
emb_sub["y"] = X_umap[:, 1]

# --------------------------------------------------
# 5. CLUSTERING
# --------------------------------------------------
emb_sub["cluster"] = KMeans(
    n_clusters=K_CLUSTERS,
    random_state=RANDOM_STATE
).fit_predict(X_pca)

# --------------------------------------------------
# 6. REMAP NODE INDICES (GLOBAL → LOCAL)
# --------------------------------------------------
node_map = {
    old: new for new, old in enumerate(emb_sub["node_id"])
}

edges_df["src_j"] = edges_df["src_i"].map(node_map)
edges_df["dst_j"] = edges_df["dst_i"].map(node_map)

# --------------------------------------------------
# 7. BUILD EDGE COORDS (PLOTLY)
# --------------------------------------------------
edge_x, edge_y = [], []

for _, r in edges_df.iterrows():
    i, j = int(r["src_j"]), int(r["dst_j"])
    edge_x += [emb_sub.iloc[i]["x"], emb_sub.iloc[j]["x"], None]
    edge_y += [emb_sub.iloc[i]["y"], emb_sub.iloc[j]["y"], None]

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode="lines",
    line=dict(width=0.4, color="rgba(0,0,0,0.15)"),
    hoverinfo="none"
)

# --------------------------------------------------
# 8. NODE TRACE
# --------------------------------------------------
node_trace = go.Scatter(
    x=emb_sub["x"],
    y=emb_sub["y"],
    mode="markers",
    hovertext=emb_sub["node_id"].astype(str),
    hoverinfo="text",
    marker=dict(
        size=7,
        color=emb_sub["cluster"],
        colorscale="Turbo",
        showscale=True,
        colorbar=dict(title="Cluster")
    )
)

# --------------------------------------------------
# 9. PLOT
# --------------------------------------------------
fig = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        title="PecanPy Embeddings — Sampled Known-Edge Graph",
        hovermode="closest",
        showlegend=False,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        margin=dict(l=20, r=20, t=40, b=20)
    )
)

fig.show()


/home/jonnym/.local/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [23]:
fig.write_html("/tmp/interactive_plot.html")